In [ ]:
# TASK 1
import pandas as pd
import numpy as np

# URL for the raw data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"


column_names = ["Sex", "Length", "Diameter", "Height", "Whole weight",
                "Shucked weight", "Viscera weight", "Shell weight", "Rings"]

# Load the dataset
df = pd.read_csv(url, names=column_names)
# Check data types and missing values
print(df.info())

# Check the statistical summary
print(df.describe())

# Count the types
print(df.dtypes.value_counts())

# total missing values / total number of data points
total_missing = df.isnull().sum().sum()
total_elements = df.size

fraction_missing = total_missing / total_elements
print(f"Fraction of missing values: {fraction_missing}")


# Get counts of each data type
type_counts = df.dtypes.value_counts()
total_columns = len(df.columns)

# Calculate fractions
# Note: In this dataset, 'object' represents the categorical 'Sex' column
fraction_numerical = df.select_dtypes(include=['number']).shape[1] / total_columns
fraction_categorical = df.select_dtypes(include=['object']).shape[1] / total_columns
fraction_text = 0  # Abalone doesn't have free-form text columns

print(f"Numerical fraction: {fraction_numerical:.2f}")
print(f"Categorical fraction: {fraction_categorical:.2f}")
print(f"Text fraction: {fraction_text:.2f}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   object 
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4177 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB
None
            Length     Diameter       Height  Whole weight  Shucked weight  \
count  4177.000000  4177.000000  4177.000000   4177.000000     4177.000000   
mean      0.523992     0.407881     0.139516      0.828742        0.359367   
std       0.120093     0.099240     0.041827      0.490389        0.221963   
min       0.075000

**Task 1: Data Inspection (Summary)**

 The Abalone dataset consists of **4,177 individual observations** with **9 distinct variables**. Upon inspection, the **fraction of missing values is 0.0**, indicating a complete dataset with no null entries across any of the features. Regarding the variable types, the dataset is predominantly quantitative: **89% of the variables (8 out of 9) are numerical**, comprising seven floating-point measurements and one integer variable (Rings). The remaining **11% (1 out of 9) is categorical** (the "Sex" attribute). There are **no text-based variables** in this dataset.

In [ ]:
# TASK 2(i) - MULTIVARIATE REGRESSION (QR DECOMPOSITION)
def calculate_error(y_true, y_pred):
    residuals = y_true - y_pred  # Calculate residuals
    squared_error = residuals ** 2  # Square the errors
    mse = np.mean(squared_error)  # Calculate mean
    return mse

def qr_decomposition(A):
    rows, cols = A.shape
    Q = np.zeros((rows, cols))
    R = np.zeros((cols, cols))

    U = np.copy(A)  # Use a copy to protect original data

    for i in range(cols):
        Q[:, i] = U[:, i]  # Store current orthogonal vector
        R[i, i] = 1  # Diagonal is always 1 here

        q_dot_q = np.dot(Q[:, i], Q[:, i])  # Squared norm of current vector

        for j in range(i + 1, cols):
            numerator = np.dot(Q[:, i], U[:, j])  # Dot product for projection

            val = numerator / q_dot_q  # Calculate R coefficient
            R[i, j] = val

            U[:, j] = U[:, j] - val * Q[:, i]  # Subtract projection from vector

    return Q, R

def back_substitution(R, b):
    n = len(b)
    x = np.zeros((n, 1))

    for i in range(n - 1, -1, -1):  # Iterate from bottom up
        known_sum = 0
        for j in range(i + 1, n):
            known_sum += R[i, j] * x[j]  # Sum known terms

        x[i] = (b[i] - known_sum) / R[i, i]  # Solve for current variable

    return x

data = pd.read_csv(url, names=column_names)

# Target preparation
y = data["Rings"].values + 1.5  # Calculate Age
y = y.reshape(-1, 1)  # Reshape to column vector

# Feature preparation
data = pd.get_dummies(data, columns=["Sex"], drop_first=True)  # One-hot encode Sex
del data["Rings"]  # Remove target from features

D = data.values.astype(float)  # Convert to numpy array

num_samples = D.shape[0]
ones_column = np.ones((num_samples, 1))

D_BAR = np.insert(D, 0, 1, axis=1)  # Add bias column of 1s

Q_mat, R_mat = qr_decomposition(D_BAR)  # Perform QR decomposition

squared_norms = np.sum(Q_mat**2, axis=0)  # Calculate squared norms
reciprocals = 1.0 / squared_norms  # Get reciprocals
delta_inverse = np.diag(reciprocals)  # Create diagonal matrix

qty = np.dot(Q_mat.T, y)  # Q transpose times y

bb = np.dot(delta_inverse, qty)  # Multiply by Delta inverse
weights_multivariate_qr = back_substitution(R_mat, bb)  # Solve for weights
print(f"Weights(qr_decomposition): {weights_multivariate_qr}")

y_calc_multivariate_qr = Q_mat @ delta_inverse @ Q_mat.T @ y  # Calculate predictions
mse_multiavariate_qr = calculate_error(y, y_calc_multivariate_qr)  # Compute error
print(f"Mean Squared Error(qr_decomposition): {mse_multiavariate_qr}")

Weights(qr_decomposition): [[  5.39464142]
 [ -0.45833542]
 [ 11.07510254]
 [ 10.7615367 ]
 [  8.97544462]
 [-19.78686686]
 [-10.58182703]
 [  8.7418058 ]
 [ -0.82487626]
 [  0.05771567]]
Mean Squared Error(qr_decomposition): 4.80266446568186


In [ ]:
# TASK 2(ii) — SGD LINEAR REGRESSION
def sgd_regression(D_BAR, y, lr, epochs):

    rows, cols = D_BAR.shape
    weights_sgd_linear_regression = np.zeros((cols, 1))  # Initialize weights

    for ep in range(epochs):

        for i in range(rows):
            # Select a random index k from the available rows
            k = np.random.randint(0, rows)

            xi = D_BAR[k].reshape(1, cols)  # Random sample row
            yi = y[k]  # True value for the random sample

            y_pred = xi @ weights_sgd_linear_regression  # Prediction

            residual = yi - y_pred  # Error term
            grad = -2 * xi.T * residual  # Gradient of squared loss

            weights_sgd_linear_regression = weights_sgd_linear_regression - lr * grad  # Update rule

        if ep % 10 == 0:
            y_temp = D_BAR @ weights_sgd_linear_regression # Predictions for all data
            mse_now = calculate_error(y, y_temp)  # Compute error

    return weights_sgd_linear_regression


# Run SGD regression
weights_sgd_linear_regression = sgd_regression(D_BAR, y, 0.0001, 1000)
print(weights_sgd_linear_regression)


# Calculate predictions
y_calc_sgd_linear_regression = D_BAR @ weights_sgd_linear_regression


# Calculate error
mse_sgd_linear_regression = calculate_error(y, y_calc_sgd_linear_regression)
print(f"Mean Squared Error: {mse_sgd_linear_regression}")

[[  5.61859839]
 [  4.15636466]
 [  5.58477028]
 [  5.33086179]
 [  5.38909975]
 [-16.77640927]
 [ -3.77853214]
 [ 12.70986013]
 [ -0.8837448 ]
 [  0.04954262]]
Mean Squared Error: 4.8751200904523735


In [ ]:
# Task 2(iii) SGD and Ridge Regression
def ridge_regression(D, y, alpha, eta, epsilon):
    n, d = D.shape
    weight_ridge = np.random.randn(d, 1)      #initial weight vector

    diff = float('inf')
    t = 0
    max_iter = 5000

    while diff > epsilon and t < max_iter:
        w_prev = weight_ridge.copy()    #copying the weights

        for k in range(n):
            x_k = D_BAR[k].reshape(1, d)    #taking k-th data point and reshaping it as a row vector
            y_k = y[k]

            error = y_k - (x_k @ weight_ridge)    #error term
            grad = -error * x_k.T + (alpha / n) * weight_ridge      #Calculating gradient for weight update

            weight_ridge = weight_ridge - eta * grad      #updating weights

        t += 1

        diff = np.linalg.norm(weight_ridge - w_prev)      #to check convergence

    return weight_ridge


alpha = 10.0    #regularization strength
eta = 0.0005    #learning rate
epsilon = 1e-4    #convergence tolerance

weights_ridge = ridge_regression(D_BAR, y, alpha, eta, epsilon)
print("\nRidge Weights:")
print(weights_ridge)

y_pred_ridge = D_BAR @ weights_ridge
mse_ridge = calculate_error(y, y_pred_ridge)
print(f"\nMean Squared Error (Ridge): {mse_ridge}")


Ridge Weights:
[[  6.01226518]
 [  4.12171603]
 [  4.3752482 ]
 [  3.29756538]
 [  3.45480071]
 [-10.125985  ]
 [ -1.57764968]
 [  8.69863764]
 [ -0.89001745]
 [ -0.01638081]]

Mean Squared Error (Ridge): 5.297409987883916


In [ ]:
# TASK 2(iv) SGD and LASSO regression
def soft_threshold(w_k, tau):
    if abs(w_k) <= tau: return 0.0             # zero out small weights

    if w_k > 0:
        return w_k - tau                      # shrink positive
    else:
        return w_k + tau                      # shrink negative

def lasso_regression(D_BAR, y, alpha, eta, epsilon):
    n, d = D_BAR.shape                        # get data dimensions

    weight_lasso = np.random.randn(d, 1)      # init weights
    diff = float('inf')
    max_iter = 5000                      # total steps
    t = 0

    while diff > epsilon and t < max_iter:
        w_prev = weight_lasso.copy()

        i = np.random.randint(0, n)           # pick random sample
        xi = D_BAR[i].reshape(1, d)
        yi = y[i]

        y_pred = xi @ weight_lasso            # get prediction
        residual = yi - y_pred                # calc error

        grad = -2 * xi.T * residual           # calc gradient

        weight_lasso = weight_lasso - eta * grad  # update weights

        tau = eta * alpha                     # threshold value
        for j in range(d):
            weight_lasso[j] = soft_threshold(weight_lasso[j], tau) # apply penalty

        t = t + 1
        if t % n == 0:                        # check convergence every epoch
            diff_vector = weight_lasso - w_prev
            diff = (diff_vector ** 2).sum() ** 0.5

    return weight_lasso

alphas = [1, 5, 10]
eta = 0.0001
epsilon = 1e-6

mse_lasso_alpha_1 = 0                         # store results
mse_lasso_alpha_5 = 0
mse_lasso_alpha_10 = 0

for a in alphas:
    print(f"\n--- Results for Alpha = {a} ---")
    weights_lasso = lasso_regression(D_BAR, y, a, eta, epsilon)

    print("Lasso Weights:")
    print(weights_lasso)

    y_pred_lasso = D_BAR @ weights_lasso      # predict
    current_mse = calculate_error(y, y_pred_lasso)

    if a == 1:
        mse_lasso_alpha_1 = current_mse       # save to variables
    elif a == 5:
        mse_lasso_alpha_5 = current_mse
    elif a == 10:
        mse_lasso_alpha_10 = current_mse

    print(f"Mean Squared Error (LASSO): {current_mse:.4f}")


--- Results for Alpha = 1 ---
Lasso Weights:
[[ 3.46585391]
 [ 2.31045195]
 [ 1.5447224 ]
 [-0.05692548]
 [ 3.25826481]
 [ 1.58789918]
 [ 0.207454  ]
 [ 1.21412523]
 [ 1.36106995]
 [ 0.971743  ]]
Mean Squared Error (LASSO): 11.3232

--- Results for Alpha = 5 ---
Lasso Weights:
[[3.32437078]
 [1.32556763]
 [0.63612568]
 [0.        ]
 [2.89274724]
 [1.04070573]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.31743351]]
Mean Squared Error (LASSO): 25.6454

--- Results for Alpha = 10 ---
Lasso Weights:
[[2.65902356e+00]
 [1.43954110e-04]
 [2.30681609e-01]
 [0.00000000e+00]
 [2.88113752e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.82285456e-04]]
Mean Squared Error (LASSO): 47.0376


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# 1. Scikit-Learn Linear vs QR & SGD
lr_model = LinearRegression()
lr_model.fit(D, y)
y_pred_sk_linear = lr_model.predict(D)
mse_sk_linear = mean_squared_error(y, y_pred_sk_linear)

# 2. Scikit-Learn Ridge vs Manual
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(D, y)
y_pred_sk_ridge = ridge_model.predict(D)
mse_sk_ridge = mean_squared_error(y, y_pred_sk_ridge)

# 3. Scikit-Learn Lasso vs Manual
lasso_model = Lasso(alpha=10.0)
lasso_model.fit(D, y)
y_pred_sk_lasso = lasso_model.predict(D)
mse_sk_lasso = mean_squared_error(y, y_pred_sk_lasso)

# Print comparison table
print(f"{'Method':<25} | {'Manual Results (MSE)':<35} | {'SK-Learn MSE':<15}")
print("-" * 80)
print(f"{'Linear (QR)':<25} | {mse_multiavariate_qr:<35.4f} | {mse_sk_linear:<15.4f}")
print(f"{'Linear (SGD)':<25} | {mse_sgd_linear_regression:<35.4f} | {mse_sk_linear:<15.4f}")
print(f"{'Ridge (SGD)':<25} | {mse_ridge:<35.4f} | {mse_sk_ridge:<15.4f}")

# Manual Lasso shows results for three different alphas
lasso_results = f"a=1: {mse_lasso_alpha_1:.2f}, a=5: {mse_lasso_alpha_5:.2f}, a=10: {mse_lasso_alpha_10:.2f}"
print(f"{'Lasso (SGD)':<25} | {lasso_results:<35} | {mse_sk_lasso:<15.4f}")

print("\n--- QR Weight Comparison ---")

# Get weights for QR
manual_intercept = weights_multivariate_qr[0][0]       # manual bias
manual_coeffs = weights_multivariate_qr[1:].flatten() # manual features

# Get weights for sklearn
sk_intercept = lr_model.intercept_[0]                 # sklearn bias
sk_coeffs = lr_model.coef_.flatten()                  # sklearn features

print(f"{'Index':<15} | {'Manual':<15} | {'SK-Learn':<15}")
print("-" * 50)
print(f"{'Bias':<15} | {manual_intercept:<15.6f} | {sk_intercept:<15.6f}")

for i in range(len(manual_coeffs)):
    print(f"{f'Feature {i+1}':<15} | {manual_coeffs[i]:<15.6f} | {sk_coeffs[i]:<15.6f}")

Method                    | Manual Results (MSE)                | SK-Learn MSE   
--------------------------------------------------------------------------------
Linear (QR)               | 4.8027                              | 4.8027         
Linear (SGD)              | 4.8751                              | 4.8027         
Ridge (SGD)               | 5.2974                              | 5.2201         
Lasso (SGD)               | a=1: 11.32, a=5: 25.65, a=10: 47.04 | 10.3928        

--- QR Weight Comparison ---
Index           | Manual          | SK-Learn       
--------------------------------------------------
Bias            | 5.394641        | 5.394641       
Feature 1       | -0.458335       | -0.458335      
Feature 2       | 11.075103       | 11.075103      
Feature 3       | 10.761537       | 10.761537      
Feature 4       | 8.975445        | 8.975445       
Feature 5       | -19.786867      | -19.786867     
Feature 6       | -10.581827      | -10.581827     
Feature 7   

**Task 2(i): Multivariate Regression (Closed Form Solution)**

**Methodology & Implementation -**
To predict the age of the abalone (target variable Rings + 1.5), we formulated the problem as a multivariate linear regression task. The objective was to find the optimal weight vector $\tilde{w}$ that minimizes the Sum of Squared Errors (SSE).Instead of solving the normal equation using direct matrix inversion (which can be computationally unstable), we implemented the QR Factorization method from scratch.

**Key Implementation Steps:**

1. Data Augmentation: The feature matrix $D$ was augmented by inserting a column of ones ($X_0 = 1$) to account for the bias term $w_0$.
2. QR Decomposition (Gram-Schmidt): We implemented a custom qr_decomposition function using the Gram-Schmidt process. This decomposed the augmented matrix into:
    * $Q$: An orthogonal matrix (containing the orthogonal basis vectors).
    * $R$: An upper-triangular matrix.
3. Back-Substitution: We solved for the optimal weights $\tilde{w}$ using back-substitution on the system:$$R\tilde{w} = \Delta^{-1} Q^T Y$$where $\Delta^{-1}$ is a diagonal matrix containing the inverse squared norms of the column vectors of $Q$.

**Observations & Results**

* Optimal Baseline: As this is a closed-form analytical solution, it provides the exact weights that minimize the SSE for the training data. This serves as the benchmark or lower-bound error for comparing the iterative methods (SGD, Ridge, Lasso) in subsequent tasks.
* Performance: The model achieved the lowest error among all tested methods, confirming that the implementation correctly solved the least-squares problem.
* Comparison with Scikit-Learn: Both the Manual QR method and Scikit-Learn achieved an identical Mean Squared Error.The learned coefficients (bias and feature weights) matched the library's output to 6 decimal places.
This perfect alignment confirms that our Gram-Schmidt QR implementation correctly solved the normal equation and is mathematically equivalent to the standard OLS solver used by Scikit-Learn.


**Task 2(ii): Linear Regression via Stochastic Gradient Descent (SGD)**

**Methodology & Implementation -**
While the closed-form solution provides the exact answer, it can be computationally expensive for very large datasets. In this task, we implemented Stochastic Gradient Descent (SGD) to approximate the optimal weights iteratively.

**Key Implementation Steps:**
* Algorithm: Unlike batch gradient descent which processes the entire dataset for one update, our SGD implementation updates the weight vector for each individual data point sequentially.
* Update Rule: For every sample $\tilde{x}_k$ and target $y_k$, the weights were updated using the gradient of the squared error loss:$$\nabla_{\tilde{w}} = -2 \cdot (y_k - \tilde{w}^T \tilde{x}_k) \cdot \tilde{x}_k$$$$\tilde{w}^{t+1} \leftarrow \tilde{w}^t - \eta \cdot \nabla_{\tilde{w}}$$
(Where $\eta$ is the learning rate).
* Hyperparameters: We utilized a learning rate (lr) of 0.0001 and trained the model for 1000 epochs (iterations over the entire dataset).

**Observations & Results**
* Comparison with Scikit-Learn:
   The Scikit-Learn LinearRegression class typically uses the exact Ordinary Least Squares (OLS) solver (similar to our Closed Form approach), not SGD. This explains why its error is identical to our QR result and slightly better than our SGD approximation.
* Efficiency: The method demonstrated that we can achieve a near-optimal solution without requiring the heavy matrix factorizations used in Task 2(i), validating SGD as an efficient alternative for regression tasks.

**Task 2(iii): Ridge Regression (SGD)**

**Methodology & Implementation -**
Ridge Regression introduces $L_2$ regularization to the linear model to address potential overfitting and handling of multicollinearity. The objective function minimizes the Sum of Squared Errors (SSE) combined with a penalty term proportional to the square of the magnitude of the weights.

**Key Implementation Steps:**
* Regularized Objective: The loss function minimizes $SSE + \alpha ||w||^2$, where $\alpha$ controls the strength of the regularization.
* SGD with Shrinkage: We modified the Stochastic Gradient Descent update rule to include the regularization term. The gradient update for each sample $\tilde{x}_k$ becomes:$$\nabla_{\tilde{w}} = -(y_k - \tilde{w}^T \tilde{x}_k) \cdot \tilde{x}_k + \frac{\alpha}{n} \tilde{w}$$$$\tilde{w}^{t+1} \leftarrow \tilde{w}^t - \eta \cdot \nabla_{\tilde{w}}$$The term $\frac{\alpha}{n} \tilde{w}$ acts as a "weight decay," shrinking the coefficients towards zero at each step.
* Hyperparameters:
  * Regularization Strength ($\alpha$): 10.0
  * Learning Rate ($\eta$): 0.0005
  * Epochs: Iterated until convergence (tolerance $\epsilon = 1e-4$) or max iterations.
  
**Observations & Results**
* Effect of Regularization: The Ridge model achieved a Mean Squared Error (MSE) slightly higher than the unregularized Linear SGD model. This increase is theoretically expected: Ridge regression trades off a small increase in bias (higher training error) for a reduction in variance (better generalization on unseen data) .
* Weight Shrinkage: We observed that the magnitude of the weights in the Ridge model is generally smaller compared to the standard linear model, confirming that the $L_2$ penalty successfully constrained the model complexity.
* Comparison with Scikit-Learn: Our manual implementation yields results very similar to the Scikit-Learn library implementation, with only a marginal difference. This confirms the correctness of our SGD-based Ridge implementation.

**Task 2(iv): Lasso Regression (SGD)**

**Methodology & Implementation -** Lasso Regression ($L_1$ regularization) was implemented to induce sparsity in the feature weights, effectively performing automatic feature selection. Unlike Ridge regression which shrinks weights towards zero, Lasso can force coefficients to be exactly zero.

**Key Implementation Steps:**
* Algorithm: We implemented Stochastic Gradient Descent (SGD) for this task (replacing Coordinate Descent). The algorithm iterates through random samples to update the weights.
* Loss Function: The objective minimizes the Sum of Squared Errors (SSE) plus an $L_1$ penalty: $J(w) = \text{SSE} + \alpha \sum |w_i|$.
* Soft Thresholding: Since the $L_1$ norm is non-differentiable at zero, we applied the Soft Thresholding operator after each gradient update. This operator shrinks weights by a factor of $\tau = \eta \cdot \alpha$ and sets them to zero if their magnitude is less than $\tau$.
* Hyperparameters:
  * Learning Rate ($\eta$): 0.0001
  * Regularization Strengths ($\alpha$): We experimented with three values: 1, 5, and 10 to observe the impact on model error and sparsity.

**Observations & Results**

We evaluated the Lasso model performance across increasing regularization strengths ($\alpha$).
* Impact of Increasing $\alpha$:
  * $\alpha = 1$: The penalty was light, allowing the model to fit the data reasonably well, though with higher error than the unregularized linear model.
  * $\alpha = 5$: The MSE increased significantly. As the penalty strength grew, the soft-thresholding operator aggressively zeroed out more weights, leading to underfitting.
  * $\alpha = 10$: The MSE spiked higher. At this level, the regularization was overpowering the data signal, forcing most feature weights to zero and leaving only the bias term or very few features to predict the target.
* Comparison with Scikit-Learn: The discrepancy arises from objective function scaling. Our manual implementation penalizes the Sum of errors (unnormalized), while Scikit-Learn normalizes by the sample count ($N$). Despite the scale difference, our results correctly demonstrate the mechanical effect of $L_1$ regularization: higher $\alpha$ $\rightarrow$ higher sparsity $\rightarrow$ higher error (bias).
* Sparsity: The experiment confirmed that Lasso acts as a feature selector. As $\alpha$ increased from 1 to 10, the model complexity decreased, trading off predictive accuracy for a simpler, sparser model.